# SPY daily contract explorer

## Tastytrade / dxFeed symbology quick reference

The collector stores **two identifiers for the same option**. Preserve both; they are not interchangeable in joins.

| Identifier | Example from the capture | Interpretation / use |
|---|---|---|
| Broker `instruments.symbol` | `SPY   261030P00758000` | OCC/OSI-style contract identifier: six-character space-padded root, expiration `YYMMDD`, `C`/`P`, eight-digit strike in thousandths. This example is a SPY put expiring 2026-10-30 at $758. |
| dxFeed `instruments.streamer_symbol` | `.SPY261030P758` | Leading dot, root, expiration `YYMMDD`, `C`/`P`, then the strike in dollars (not scaled or zero-padded; fractional strikes can contain a decimal point). Used for stream subscriptions. |
| Massive equivalent | `O:SPY261030P00758000` | `O:` prefix, unpadded root, and the same date/type/eight-digit scaled strike as the broker identifier. |
| Underlying | `SPY` | The stock/ETF identifier, without an option prefix. Both instrument identifiers are `SPY`; option type, strike and expiration are null. |

For the broker example, `00758000 / 1000 = 758.000`; the three spaces after `SPY` complete the six-character root. For these standard SPY contracts, removing the root padding and adding `O:` gives the Massive identifier. Do **not** simply add `O:` to a dxFeed identifier: its strike encoding differs.

`quotes.symbol`, `trades.symbol`, `analytics.symbol`, `samples.symbol`, and `raw_updates.symbol` use the **streamer symbol**. Join them to `instruments.streamer_symbol`, not the broker `symbol`. `analytics` stores broker/feed Greeks and implied volatility; these are not trade-based option aggregates or independently calculated Greeks.

Use the recorded `expiration`, `option_type`, `strike`, and `shares_per_contract` columns as contract terms instead of reparsing ticker text. The current selector checks SPY root and 100 shares per contract; that check alone does not verify every deliverable detail. This reference covers the standard SPY options collected here, not every tastytrade instrument type.

## Choose one recorded day and connect locally

Purpose: understand which contracts were selected and what was actually recorded on each day, so collection changes can be informed by evidence. This initial notebook connects to the local capture and lists contracts; it does not yet evaluate intraday freshness, gaps or model suitability.

- **Stop the collector before running.** DuckDB access is read-only and each query closes its connection. No API connection, credentials, or orders are involved.
- Select the standard-Python **`.venv-collector`** kernel. Notebook dependencies are in `requirements-notebook.in`.
- Set `ANALYSIS_DAY` below to an ISO date. The default, **2026-10-30**, is the first recorded day in the current capture. Set it to `None` to discover the earliest session with recorded instrument metadata in another capture.
- A day means the collector's **New York exchange `runs.session_date`**, not the UTC date of an individual update. Include **all runs for that day**, including restarts and partial/failed runs.
- Keep this source notebook free of saved outputs. Save an executed copy under the Git-ignored `exports/notebooks/` directory if desired.

Times below are displayed in UTC. Source event timestamps can be unknown; receipt timestamps are local arrival times, not exchange event times.

In [ ]:
from datetime import date
from pathlib import Path
import sys

import duckdb
import pandas as pd
from IPython.display import display

ANALYSIS_DAY = "2026-10-30"  # Use None for the earliest recorded session.

repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "Tastyware_Demo_Scripts" / "collector_core.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run this notebook from inside the Tasty_Dearcyfi_Tests repository.")

DATABASE_PATH = repo_root / "Tastyware_Demo_Scripts" / "exports" / "spy_validation.duckdb"
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f"Capture database does not exist: {DATABASE_PATH}")

def query(sql: str, parameters: list[object] | None = None) -> pd.DataFrame:
    with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
        connection.execute("SET TimeZone='UTC'")
        return connection.execute(sql, parameters if parameters is not None else []).df()

schema_versions = query("SELECT version FROM collector_schema")["version"].tolist()
if schema_versions != [2]:
    raise ValueError(f"Expected collector schema 2; found {schema_versions}.")

recorded_days = query("""
    SELECT r.session_date, count(DISTINCT r.run_id) AS run_count,
           count(DISTINCT i.streamer_symbol) FILTER (
               WHERE i.option_type IS NOT NULL
           ) AS option_contracts
    FROM runs r LEFT JOIN instruments i USING (run_id)
    GROUP BY r.session_date ORDER BY r.session_date
""")
if recorded_days.empty:
    raise ValueError("The capture contains no collection runs.")
if ANALYSIS_DAY is None:
    first_recorded = query("""
        SELECT min(r.session_date) AS session_date
        FROM runs r JOIN instruments i USING (run_id)
    """)["session_date"].iloc[0]
    if pd.isna(first_recorded):
        raise ValueError("No instrument metadata has been recorded in this capture.")
    selected_day = first_recorded.date()
else:
    selected_day = date.fromisoformat(ANALYSIS_DAY)
available_days = recorded_days["session_date"].dt.date.tolist()
if selected_day not in available_days:
    raise ValueError(f"No runs for {selected_day}; available dates: {available_days}.")

pd.set_option("display.max_columns", 24)
pd.set_option("display.max_colwidth", 100)
print(f"Python: {sys.executable}")
print(f"Database: {DATABASE_PATH} (read-only; schema 2)")
print(f"Selected New York session: {selected_day}")
display(recorded_days)

## Daily universe and run provenance

The universe is dynamic, but it is **cumulative within a session**, not replaced on every refresh:

- The current collector refreshes from the option chain and a fresh SPY midpoint every **1,800 seconds (30 minutes) by default**. The saved run configuration below records the actual interval and cap for each run.
- Default expiration targets are 0, 7, 14, 17, 28 and 56 calendar days, mapped to the nearest eligible listed expirations. It selects up to nine puts around strike/spot anchors 0.95, 0.975 and 1.0, plus three calls nearest spot per chosen expiration.
- Newly selected contracts are added; previously selected contracts remain subscribed for the session. If an expansion would exceed the default **96-option cumulative cap**, the whole expansion is rejected, not partially added. SPY is subscribed separately.
- Same-day restarts restore previously retained contracts from this database. A later session selects anew, so daily contract lists can differ as spot and eligible expirations change.

The contract list below is the **union across the day's runs**, not just the opening selection or one revision. A metadata selection is not confirmation of successful subscription or continuous feed coverage. Run status and revision counts provide context, not a quality certificate.

The first day contains both schema-1 and schema-2 runs. Schema-1 normalized observations were sampled; schema-2 observations retain received updates. Upgrading the database does not recreate schema-1 update history.

In [ ]:
daily_runs = query("""
    SELECT r.run_id, r.schema_version, r.started_at, r.ended_at,
           r.status, r.stop_reason,
           json_extract_string(r.capabilities, '$.recording_mode') AS recording_mode,
           json_extract_string(r.config, '$.refresh_seconds') AS refresh_seconds,
           json_extract_string(r.config, '$.max_contracts') AS max_contracts,
           (SELECT count(*) FROM universe_revisions u
            WHERE u.run_id=r.run_id) AS universe_revision_records
    FROM runs r WHERE r.session_date=? ORDER BY r.started_at, r.run_id
""", [selected_day])
display(daily_runs)

## Contracts recorded for the selected day

One row per option **streamer symbol**, with the latest recorded metadata for that day. Repeated universe revisions and restored metadata do not multiply the contract count. `first_selected_at_utc` is the earliest persisted selection across that day's runs; `metadata_run_count` counts runs containing that contract's metadata, not successful connections.

`quote_rows`, `trade_rows`, and `analytics_rows` count **normalized stored observations** across the day's runs, joined by streamer symbol. These are not unique economic events, minute bars, or proof of timestamp freshness; schema-1 and schema-2 counts have different recording semantics. Zero means no normalized observation of that kind was stored, even if metadata was selected. The underlying SPY is excluded from the option list.

The complete contract table is displayed without row truncation, followed by a compact expiration/type inventory.

In [ ]:
contracts = query("""
    WITH day_runs AS (
        SELECT run_id FROM runs WHERE session_date=?
    ), day_instruments AS (
        SELECT i.* FROM instruments i JOIN day_runs USING (run_id)
        WHERE i.option_type IS NOT NULL
    ), selection_history AS (
        SELECT streamer_symbol, min(effective_at) AS first_selected_at_utc,
               count(DISTINCT run_id) AS metadata_run_count
        FROM day_instruments GROUP BY streamer_symbol
    ), latest_metadata AS (
        SELECT * FROM day_instruments
        QUALIFY row_number() OVER (
            PARTITION BY streamer_symbol
            ORDER BY effective_at DESC, received_at_utc DESC, record_id DESC
        )=1
    ), quote_counts AS (
        SELECT q.symbol AS streamer_symbol, count(*) AS quote_rows
        FROM quotes q JOIN day_runs USING (run_id) GROUP BY q.symbol
    ), trade_counts AS (
        SELECT t.symbol AS streamer_symbol, count(*) AS trade_rows
        FROM trades t JOIN day_runs USING (run_id) GROUP BY t.symbol
    ), analytics_counts AS (
        SELECT a.symbol AS streamer_symbol, count(*) AS analytics_rows
        FROM analytics a JOIN day_runs USING (run_id) GROUP BY a.symbol
    )
    SELECT i.symbol AS broker_symbol, i.streamer_symbol, i.expiration,
           i.option_type, i.strike, i.shares_per_contract,
           i.exercise_style, i.settlement_type,
           h.first_selected_at_utc, h.metadata_run_count,
           coalesce(q.quote_rows, 0) AS quote_rows,
           coalesce(t.trade_rows, 0) AS trade_rows,
           coalesce(a.analytics_rows, 0) AS analytics_rows
    FROM latest_metadata i JOIN selection_history h USING (streamer_symbol)
    LEFT JOIN quote_counts q USING (streamer_symbol)
    LEFT JOIN trade_counts t USING (streamer_symbol)
    LEFT JOIN analytics_counts a USING (streamer_symbol)
    ORDER BY i.expiration, i.option_type, i.strike, i.streamer_symbol
""", [selected_day])

print(f"{selected_day}: {len(contracts)} distinct option contracts across {len(daily_runs)} runs.")
if contracts.empty:
    raise ValueError(f"Runs exist for {selected_day}, but no option instrument metadata was recorded.")
with pd.option_context("display.max_rows", None):
    display(contracts)

expiration_inventory = contracts.groupby(["expiration", "option_type"], as_index=False).agg(
    contract_count=("streamer_symbol", "nunique"),
    minimum_strike=("strike", "min"),
    maximum_strike=("strike", "max"),
)
display(expiration_inventory)